# [8.1] Activation Patching Refresher - Solutions

This notebook validates the helper contracts, inspects the committed CUDA signature result, and then reruns the live TransformerLens activation-patching preflight through `solutions.py`.

<details>
<summary>Expected output</summary>

All local tests should pass. The signature result should show patch scores `[0, 0, 0, 0, 0, 1]`, target recovery `1.0`, wrong-position mean/max `0.0`, and a live CUDA run on `torch 2.12.1+cu132`.

</details>

<details>
<summary>Help - why rerun live CUDA?</summary>

The committed report is the review artifact, but the live cell proves the current `uv` environment, GPU, TransformerLens loader, tokenizer revision, and patching hook still work together.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t

chapter = "chapter8_automated_circuits"
section = "part1_activation_patching_refresher"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_activation_patching_refresher.tests as tests
import part1_activation_patching_refresher.utils as utils

from part1_activation_patching_refresher import solutions


## Unit Contracts

The tests are deliberately small. They catch metric direction, in-place patching, degenerate recovery, bad index handling, and the max wrong-control requirement before the live model path.

<details>
<summary>Help - why not only test the CUDA report?</summary>

A single CUDA boolean is too coarse for debugging. If activation patching fails, you need to know whether the bug is the readout, the patching slice, the recovered-fraction normalization, or the localization/control logic.

</details>


In [ ]:
tests.test_answer_logit_diff_validates_token_ids(solutions.answer_logit_diff)
tests.test_answer_logit_diff_rejects_degenerate_metrics(solutions.answer_logit_diff)
tests.test_patch_activation_slice_replaces_one_component_without_mutating_inputs(
    solutions.patch_activation_slice,
)
tests.test_patching_recovery_report_and_sweep_normalize_by_clean_corrupt_gap(
    solutions.patching_recovery_report,
    solutions.activation_patching_sweep,
    solutions.recovery_fraction,
)
tests.test_recovery_and_sweep_reject_degenerate_inputs(
    solutions.patching_recovery_report,
    solutions.activation_patching_sweep,
)
tests.test_localization_and_random_controls_require_top_components_to_win(
    solutions.patching_localization_report,
    solutions.random_patch_control_report,
)
tests.test_localization_and_random_controls_reject_bad_indices(
    solutions.patching_localization_report,
    solutions.random_patch_control_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


## CPU Contract

Before the live model, the smoke report should already have the section shape: scalar metric, clean slice patch, recovered fraction, sweep, localization, and wrong-position controls.

<details>
<summary>Expected output</summary>

`logit_diff` should be `2.0`, recovered fraction should be `0.8`, sweep scores should be `[0.2, 0.8, 0.4]`, and `top_beats_max_random` should be `True`.

</details>

<details>
<summary>Common bug</summary>

If `best_index` is correct but `patch_scores` are raw logits rather than recovered fractions, the CUDA patching sweep will be hard to compare across prompts.

</details>


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["logit_diff"] == 2.0
assert contract["patch_slice"] == [[0.0, 0.0], [30.0, 40.0]]
assert contract["recovery"]["recovered_fraction"] == 0.8
assert contract["sweep"]["patch_scores"] == [0.2, 0.8, 0.4]
assert contract["localization"]["localizes_target"]
assert contract["random_control"]["top_beats_random"]
assert contract["random_control"]["top_beats_max_random"]
utils.print_report(
    "CPU smoke contract",
    {
        "logit_diff": contract["logit_diff"],
        "patch_slice": contract["patch_slice"],
        "recovered_fraction": contract["recovery"]["recovered_fraction"],
        "patch_scores": contract["sweep"]["patch_scores"],
        "top_indices": contract["localization"]["top_indices"],
        "max_random_patch_score": contract["random_control"]["max_random_patch_score"],
    },
)


## Signature Result

Now inspect the accepted CUDA report. The important result is the pattern of patching scores, not only the final boolean.

<details>
<summary>Interpreting the signature result</summary>

Final-position patching recovers exactly because the patched activation is the final residual stream immediately before the unembedding. The non-final controls staying at zero show the result is not caused by arbitrary clean activations. The result validates patching mechanics, not an upstream circuit graph.

</details>


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"] and report["tests_passed"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["target_token"] == " floor"
assert gpu["distractor_token"] == " top"
assert gpu["patch_scores_by_position"] == [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
assert gpu["best_position"] == gpu["target_position"] == 5
assert gpu["target_recovered_fraction"] >= 0.99
assert abs(gpu["wrong_position_control_fraction"]) <= 1e-4
assert abs(gpu["max_wrong_position_control_fraction"]) <= 1e-4
assert gpu["top_beats_wrong_position_control"]
assert gpu["top_beats_max_wrong_position_control"]

fig, ax = plt.subplots(figsize=(6, 3))
positions = list(range(len(gpu["patch_scores_by_position"])))
ax.bar(positions, gpu["patch_scores_by_position"], color=["#6b7280"] * 5 + ["#2563eb"])
ax.set_xlabel("patched residual position")
ax.set_ylabel("recovered fraction")
ax.set_ylim(-0.05, 1.1)
ax.set_title("Activation patching recovery by position")
ax.axhline(0, color="black", linewidth=0.8)
plt.show()

utils.print_report(
    "Committed CUDA signature result",
    {
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "device": gpu["device"],
        "clean_metric": round(gpu["clean_metric"], 4),
        "corrupt_metric": round(gpu["corrupt_metric"], 4),
        "target_recovered_fraction": gpu["target_recovered_fraction"],
        "max_wrong_position_control": gpu["max_wrong_position_control_fraction"],
        "peak_vram_gb": round(gpu["peak_vram_gb"], 4),
    },
)


## Live CUDA Path

The report above is committed evidence. This cell reruns the live CUDA path on the current machine through `solutions.py`.

<details>
<summary>Expected output</summary>

`preflight_passed` should be `True`, peak VRAM should stay below the 24GB budget, and the live patch scores should match `[0, 0, 0, 0, 0, 1]`.

</details>


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_full_experiment(max_vram_gb=max_vram_gb)


live_gpu = run_full_experiment(max_vram_gb=24.0)
assert live_gpu["preflight_passed"]
assert live_gpu["patch_scores_by_position"] == [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
assert live_gpu["best_position"] == live_gpu["target_position"] == 5
assert live_gpu["target_recovered_fraction"] >= 0.99
assert abs(live_gpu["max_wrong_position_control_fraction"]) <= 1e-4
assert live_gpu["top_beats_max_wrong_position_control"]
assert live_gpu["peak_vram_gb"] <= 24.0
utils.print_report(
    "Live CUDA preflight",
    {
        "torch": live_gpu["torch_version"],
        "cuda": live_gpu["cuda_version"],
        "device": live_gpu["device"],
        "patch_scores": live_gpu["patch_scores_by_position"],
        "target_recovered_fraction": live_gpu["target_recovered_fraction"],
        "max_wrong_position_control": live_gpu["max_wrong_position_control_fraction"],
        "peak_vram_gb": round(live_gpu["peak_vram_gb"], 4),
    },
)


## Limitations

This is a GT-1 activation-patching mechanics preflight on one pinned `gelu-1l` hook and one safe prompt pair. It is not a full IOI circuit analysis, not an OOD prompt-template benchmark, and not a feature-level attribution graph.

## Further Research

Repeat the sweep across prompt templates, compare denoising and noising patches, patch blocks/heads/QKV/patterns, and use this contract as the baseline for attribution patching, ACDC, EAP, and sparse feature circuits in the next Chapter 8 sections.
